# PianoVAM v1.2 - un singolo esempio, passo passo

Questo notebook prende il dataset **gia' scaricato** in locale (o su Google Drive) e mostra
**tutte le informazioni di un singolo esempio**: la riga di `metadata.json`, la riga di nota
del `TSV/`, la riga corrispondente di `Fingering/`, il MIDI, lo scheletro delle mani,
l'audio e il video.

Dataset: [`PianoVAM/PianoVAM_v1`](https://huggingface.co/datasets/PianoVAM/PianoVAM_v1)
(107 registrazioni, ~45 GB, licenza **CC BY-NC-SA 4.0**).

## Come si usa

Esegui le celle dall'alto in basso. La cella **DATASET** piu' sotto sistema tutto da sola:

- se il dataset **non c'e'**, scarica dal Hub **una sola registrazione** (i file testuali
  pesano pochi KB, quindi parte in pochi secondi);
- se il dataset **c'e' gia'** (anche su Google Drive) non scarica niente e non usa la rete.

Non serve un ordine preciso: se la cella DATASET non e' stata eseguita, la cella che legge
`metadata.json` scarica da sola la registrazione che le serve; se non riesce (per esempio
senza rete), ti stampa il motivo e le due strade possibili.

Se hai il dataset su Google Drive, monta il Drive e cambia `DATASET_ROOT` nella cella 0
(CONFIGURAZIONE): le righe sono gia' pronte e commentate (`from google.colab import drive` /
`drive.mount("/content/drive")` / `DATASET_ROOT = "/content/drive/MyDrive/PianoVAM_v1.2"`).

Audio, video e scheletro delle mani pesano centinaia di MB, quindi **arrivano solo quando
servono**: la cella dell'audio, quella del video e quella dello scheletro scaricano da sole il
file che manca (`AUTO_DOWNLOAD_MEDIA = True`), senza pre-scaricare 500 MB all'inizio. Il video
lo vedi come **estratto di 15 s a 640 px** (`VIDEO_EXCERPT_SECONDS`), perche' 324 MB dentro
l'output del notebook non si possono mettere; il file completo resta sul disco e il percorso e'
stampato. Con `SHOW_VIDEO = False` / `SHOW_AUDIO_EXCERPT = False` salti le anteprime,
`LOAD_HAND_SKELETON = True` analizza il JSON delle mani, mentre `DOWNLOAD_MEDIA = True` /
`DOWNLOAD_SKELETON = True` pre-scaricano tutto nella cella DATASET.

Serve solo `pandas` (in Colab c'e' gia'). Per i dettagli del MIDI serve `mido`, per l'estratto
audio e per l'estratto video serve `ffmpeg` (con `ffprobe`): sono opzionali e in Colab `ffmpeg`
e' preinstallato. Se `ffmpeg` mancasse, la cella video mostra comunque i dati tecnici del file e
ti dice dove trovarlo.

Se il dataset non e' ancora sul disco, scaricalo una volta sola da terminale:

```bash
pip install huggingface_hub
huggingface-cli download PianoVAM/PianoVAM_v1 --repo-type dataset --local-dir ./PianoVAM_v1.2
```

Oppure lascia fare al notebook: la cella DATASET (o direttamente la cella del `metadata.json`)
scarica **una sola registrazione**, pochi KB senza audio e video.

In [ ]:
# ======================================================================
# 0) CONFIGURAZIONE
# ======================================================================
# Il notebook lavora sul dataset GIA' SCARICATO: basta indicare la cartella
# che contiene metadata.json e le sottocartelle Audio/ Video/ MIDI/ TSV/ ...

from pathlib import Path
from glob import glob
from collections import Counter
import datetime
import json
import subprocess
import tempfile
import wave

import pandas as pd

DATASET_ROOT = "PianoVAM_v1.2"   # cartella del dataset (relativa o assoluta)
# Se invece hai il dataset su Google Drive, in Colab monta il Drive e cambia la riga sopra:
#   from google.colab import drive
#   drive.mount("/content/drive")
#   DATASET_ROOT = "/content/drive/MyDrive/PianoVAM_v1.2"

RECORD_INDEX = 0          # indice 0..106 nella metadata.json
RECORD_TIME = None        # es. "2024-02-14_19-10-09": prevale su RECORD_INDEX

# Il dataset e' pesante: carichiamo solo cio' che serve
LOAD_HAND_SKELETON = False   # Handskeleton/*.json pesa ~120 MB per registrazione
SHOW_AUDIO_EXCERPT = True    # crea un estratto di 8 s dell'audio e lo riproduce
SHOW_VIDEO = True            # mostra il primo estratto del video (il file intero e' ~340 MB)
EXCERPT_SECONDS = 8          # secondi di audio da ascoltare
VIDEO_EXCERPT_SECONDS = 15   # secondi di video da mostrare (ridimensionato, pochi MB)

# Cosa scaricare dal Hub: i file testuali pesano pochi KB, i media centinaia di MB
DOWNLOAD_ONE_RECORDING = True    # cella DATASET: scarica TSV, MIDI e Fingering (pochi KB)
DOWNLOAD_MEDIA = False           # cella DATASET: pre-scarica anche Audio + Video (~390 MB)
DOWNLOAD_SKELETON = False        # cella DATASET: pre-scarica anche Handskeleton (~120 MB)
AUTO_DOWNLOAD_MEDIA = True       # celle Audio/Video/Scheletro: si prendono da sole il file che
                                 # manca, quando le esegui (niente pre-download da 500 MB)
REPO_HUB = "PianoVAM/PianoVAM_v1"

LIBRO = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def pitch_name(note_number):
    """Nome della nota a partire dal numero MIDI (60 -> C4)."""
    n = int(note_number)
    return LIBRO[n % 12] + str(n // 12 - 1)


def human(size_bytes):
    """Dimensione leggibile."""
    value = float(size_bytes)
    for unit in ["B", "KB", "MB", "GB"]:
        if value < 1024:
            return "%.1f %s" % (value, unit)
        value /= 1024
    return "%.1f TB" % value


def read_tsv(path, dtype=None):
    """Legge un file TSV del dataset: la prima riga e' sempre l'header.

    In TSV/ l'header e' commentato con '#', in Fingering/ e Fingering_GT/ no,
    quindi lo leggiamo a mano e passiamo i nomi delle colonne a pandas.
    """
    with open(path, encoding="utf-8") as handle:
        intestazione = handle.readline()
    colonne = [campi.strip().lstrip("#").strip() for campi in intestazione.split("\t")]
    tabella = pd.read_csv(path, sep="\t", skiprows=1, names=colonne, dtype=dtype)
    # gli stessi tipi del dataset su Hugging Face: 3 float64 + 2 int64
    for colonna in ["onset", "key_offset", "frame_offset"]:
        if colonna in tabella:
            tabella[colonna] = tabella[colonna].astype(float)
    for colonna in ["note", "velocity"]:
        if colonna in tabella:
            tabella[colonna] = tabella[colonna].astype(int)
    return tabella


def title(text):
    print()
    print("=" * 78)
    print(text)
    print("=" * 78)


def find_dataset_root(configured):
    """Restituisce la cartella del dataset: quella che contiene metadata.json."""
    candidates = [Path(configured),
                  Path.cwd() / "PianoVAM_v1.2",
                  Path.cwd() / "PianoVAM_v1",
                  Path("/content/PianoVAM_v1.2"),
                  Path("/content/drive/MyDrive/PianoVAM_v1.2"),
                  Path("/content/drive/MyDrive/datasets/PianoVAM_v1.2"),
                  Path.home() / "PianoVAM_v1.2"]
    # eventuale snapshot lasciato dalla cache di huggingface_hub
    cache = Path.home() / ".cache/huggingface/hub/datasets--PianoVAM--PianoVAM_v1/snapshots"
    candidates += [Path(p) for p in glob(str(cache / "*"))]
    for candidate in candidates:
        if (candidate / "metadata.json").is_file():
            return candidate.resolve()
    cercate = ["  - %s" % percorso for percorso in candidates[:7]]
    raise FileNotFoundError(
        "metadata.json non trovato: ho cercato in\n" + "\n".join(cercate) + "\n"
        "Due strade:\n"
        "  1) hai il dataset su Google Drive: monta il Drive e metti in cima al notebook\n"
        "     DATASET_ROOT = \"/content/drive/MyDrive/PianoVAM_v1.2\"\n"
        "  2) non hai ancora il dataset: lascia DOWNLOAD_ONE_RECORDING = True in questa cella\n"
        "     e scarica UNA registrazione (pochi KB). Se hai appena provato e non e' riuscito,\n"
        "     il motivo e' stampato appena sopra questo messaggio.")


def scarica_registrazione(root, record_time=None):
    """Scarica dal Hub UNA registrazione del dataset (i file testuali pesano pochi KB).

    Non riscarica mai lo stesso file: se ci sono gia', lascia tutto com'e'.
    Restituisce True se i file necessari della registrazione sono sul disco.
    """
    try:
        from huggingface_hub import hf_hub_download, snapshot_download
    except ImportError:
        print("Serve huggingface_hub: esegui  !pip install -q huggingface_hub  e riprova.")
        return False

    root = Path(root)
    try:
        # 1) metadata.json e' l'elenco delle 107 registrazioni (66 KB)
        locale = root / "metadata.json"
        if locale.is_file():
            meta_all = json.loads(locale.read_text(encoding="utf-8"))
        else:
            meta_file = hf_hub_download(REPO_HUB, "metadata.json", repo_type="dataset",
                                        local_dir=str(root))
            meta_all = json.loads(Path(meta_file).read_text(encoding="utf-8"))
        wanted = record_time or meta_all[sorted(meta_all, key=int)[RECORD_INDEX]]["record_time"]

        # 2) i file di quella registrazione
        richiesti = ["TSV/%s.tsv" % wanted, "MIDI/%s.mid" % wanted]
        if DOWNLOAD_SKELETON:
            richiesti.append("Handskeleton/%s.json" % wanted)
        if DOWNLOAD_MEDIA:
            richiesti += ["Audio/%s.wav" % wanted, "Video/%s.mp4" % wanted]
        # Fingering/ non esiste per la registrazione a 4 mani e Fingering_GT/ c'e' solo per
        # 11: li chiediamo senza pretendere che ci siano (il Hub salta quelli che non ci sono).
        facoltativi = ["Fingering/%s.tsv" % wanted, "Fingering_GT/%s.tsv" % wanted]

        mancanti = [nome for nome in richiesti if not (root / nome).is_file()]
        if not mancanti:
            print("Registrazione", wanted, ": i file necessari ci sono gia' in", root,
                  "- non scarico nulla.")
        else:
            print("Scarico in", root, ":", ", ".join(mancanti + facoltativi))
            snapshot_download(REPO_HUB, repo_type="dataset", local_dir=str(root),
                              allow_patterns=mancanti + facoltativi)
            print("Fatto.")
    except Exception as errore:
        print("Download non riuscito (%s): %s" % (type(errore).__name__, errore))
        print("Su Colab, se la rete non funziona, monta Google Drive e metti in DATASET_ROOT")
        print("la copia del dataset che hai li' (righe pronte in cima a questa cella).")
        return False

    if not (DOWNLOAD_MEDIA and DOWNLOAD_SKELETON):
        print("Audio, video e scheletro non sono ancora sul disco: non serve scaricarli adesso,")
        print("perche' le celle che li mostrano se li prendono da sole (AUTO_DOWNLOAD_MEDIA).")
        print("Se li vuoi subito tutti: DOWNLOAD_MEDIA = True e DOWNLOAD_SKELETON = True.")
    return True


def file_modalita(root, relativo, peso_mb=None):
    """Path del file di una modalita'; se manca e AUTO_DOWNLOAD_MEDIA e' True lo scarica.

    E' il modo "pigro": la cella dell'audio, del video e dello scheletro prendono da sole il
    file che serve, solo quando le esegui. Restituisce None se il file non c'e' e non si e'
    potuto scaricare (spiegando il motivo), cosi' la cella chiamante salta la modalita'.
    """
    percorso = Path(root) / relativo
    if percorso.is_file():
        return percorso
    if not AUTO_DOWNLOAD_MEDIA:
        print("%s non presente in %s" % (relativo, root))
        print("Per scaricarlo: AUTO_DOWNLOAD_MEDIA = True nella cella 0 e riesegui questa cella,")
        print("oppure DOWNLOAD_MEDIA = True / DOWNLOAD_SKELETON = True e riesegui la cella DATASET.")
        return None

    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("Serve huggingface_hub: esegui  !pip install -q huggingface_hub  e riprova.")
        return None

    peso = "" if peso_mb is None else " (~%s MB)" % peso_mb
    print("Scarico %s dal Hub%s in %s ..." % (relativo, peso, root))
    try:
        hf_hub_download(REPO_HUB, relativo, repo_type="dataset", local_dir=str(root))
    except Exception as errore:
        print("Download non riuscito (%s): %s" % (type(errore).__name__, errore))
        print("Se sei su Colab senza rete, monta Google Drive e metti DATASET_ROOT sulla copia")
        print("del dataset che hai li' (righe pronte in cima a questa cella).")
        return None
    if not percorso.is_file():
        print("Il Hub non ha %s: questa registrazione non ha quella modalita'." % relativo)
        return None
    print("Fatto:", human(percorso.stat().st_size))
    return percorso

In [ ]:
# ======================================================================
# 1) DATASET: se non c'e' ancora, scaricalo dal Hub (una registrazione)
# ======================================================================
# Si puo' eseguire piu' volte: scarica solo i file che mancano, quindi se il dataset
# e' gia' al suo posto (anche su Google Drive) non scarica nulla e non usa la rete.
# Se salti questa cella non e' un problema: la cella del metadata la esegue da sola.
# Per il dataset completo (~45 GB, video inclusi), da terminale:
#   huggingface-cli download PianoVAM/PianoVAM_v1 --repo-type dataset --local-dir ./PianoVAM_v1.2

if DOWNLOAD_ONE_RECORDING:
    scarica_registrazione(DATASET_ROOT, RECORD_TIME)
else:
    print("DOWNLOAD_ONE_RECORDING = False: uso solo i file gia' presenti in", DATASET_ROOT)

In [ ]:
# ======================================================================
# 2) LA RIGA DEL SINGOLO ESEMPIO: metadata.json
# ======================================================================
# Se il dataset non c'e' ancora, questa cella se lo scarica da sola (una registrazione):
# cosi' non devi eseguire le celle in un ordine preciso.
if "scarica_registrazione" not in globals():
    raise RuntimeError("Manca la configurazione: esegui prima la cella 0 (CONFIGURAZIONE).")

try:
    ROOT = find_dataset_root(DATASET_ROOT)
except FileNotFoundError as errore_mancante:
    if not DOWNLOAD_ONE_RECORDING:
        raise
    print("Il dataset non c'e' ancora in", DATASET_ROOT, ": lo scarico dal Hub ...")
    if not scarica_registrazione(DATASET_ROOT, RECORD_TIME):
        print()
        print(errore_mancante)
        raise
    ROOT = find_dataset_root(DATASET_ROOT)
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
row_keys = sorted(metadata, key=int)   # metadata.json e' un dict: {"0": {...}, "1": {...}}

print("Cartella dataset :", ROOT)
print("Registrazioni    :", len(row_keys))
print("Split            :", dict(Counter(v["split"] for v in metadata.values())))

if RECORD_TIME is not None:
    row_key = next(k for k in row_keys if metadata[k]["record_time"] == RECORD_TIME)
else:
    row_key = row_keys[RECORD_INDEX]

record = metadata[row_key]
record_time = record["record_time"]   # id usato nei nomi file di tutte le sottocartelle

title("RIGA %s di metadata.json  ->  %s" % (row_key, record_time))
print("Tutti i campi di questa registrazione:")
for field, value in record.items():
    print("  %-18s: %s" % (field, value))

In [ ]:
# ======================================================================
# 3) LE NOTE: TSV/<record_time>.tsv   (una riga = una nota)
# ======================================================================
tsv_path = ROOT / "TSV" / (record_time + ".tsv")
notes = read_tsv(tsv_path)   # una riga = una nota; la prima riga del file e' l'header

title("TSV/%s  ->  %d note" % (tsv_path.name, len(notes)))
print("Colonne:", list(notes.columns))
print("Prime 5 righe:")
print(notes.head().to_string(index=False))

i = 0                     # quale riga di nota esaminare in dettaglio
nota = notes.iloc[i]
title("UNA RIGA COMPLETA (riga %d di %d)" % (i + 1, len(notes)))
print("  onset        : %-12s s dall'inizio (il tasto scende)" % nota["onset"])
print("  key_offset   : %-12s s in cui il dito lascia fisicamente il tasto" % nota["key_offset"])
print("  frame_offset : %-12s s in cui il suono finisce (con pedale)" % nota["frame_offset"])
print("  note         : %-12d numero MIDI  ->  %s" % (nota["note"], pitch_name(nota["note"])))
print("  velocity     : %-12d forza del tasto, da 1 a 127" % nota["velocity"])
print()
print("  durata del tasto : %.4f s" % (nota["key_offset"] - nota["onset"]))
print("  durata del suono : %.4f s" % (nota["frame_offset"] - nota["onset"]))

print()
print("Statistiche di tutte le note della registrazione:")
print(notes.describe().to_string())
print()
print("Altezze: %s - %s (MIDI %d - %d)" % (pitch_name(notes["note"].min()),
                                          pitch_name(notes["note"].max()),
                                          notes["note"].min(), notes["note"].max()))

In [ ]:
# ======================================================================
# 4) LA DITEGGIATURA: Fingering/<record_time>.tsv
#    (stesse righe di TSV/, con in piu' hand e finger)
# ======================================================================
fing_path = ROOT / "Fingering" / (record_time + ".tsv")
if fing_path.is_file():
    fingering = read_tsv(fing_path, dtype={"hand": str, "finger": str})
    assert len(fingering) == len(notes)   # allineate 1:1 con TSV/

    title("Fingering/%s  ->  %d note" % (fing_path.name, len(fingering)))
    print("Le prime 5 colonne sono identiche a TSV/, le altre due sono:")
    print("  hand   : L (mano sinistra), R (mano destra) oppure Noinfo")
    print("  finger : 1 (pollice) .. 5 (mignolo) oppure Noinfo")
    print()
    print("La riga %d di questa registrazione, con tutte le colonne:"
          % (i + 1))
    for column in fingering.columns:
        print("  %-13s: %s" % (column, fingering[column].iloc[i]))
    print()
    labeled = fingering[fingering["hand"] != "Noinfo"]
    print("Note con diteggiatura: %d/%d (%.1f%%)" % (len(labeled), len(fingering),
                                                     100.0 * len(labeled) / len(fingering)))
    print("Per mano:")
    print(fingering["hand"].value_counts().to_string())
    print("Per dito:")
    print(fingering["finger"].value_counts().to_string())
else:
    print("Fingering/%s.tsv assente: e' la registrazione a 4 mani, le etichette"
          " di diteggiatura esistono solo per i soli." % record_time)

gt_path = ROOT / "Fingering_GT" / (record_time + ".tsv")
if gt_path.is_file():
    gt = read_tsv(gt_path, dtype={"hand": str, "finger": str})
    title("Fingering_GT/%s  ->  %d note annotate a mano" % (gt_path.name, len(gt)))
    print(gt.head(5).to_string(index=False))
    print("(Fingering_GT/ copre 11 registrazioni: le prime 150/300 note di ciascuna.)")
else:
    print("Fingering_GT/%s.tsv assente (esiste solo per 11 registrazioni)." % record_time)

In [ ]:
# ======================================================================
# 5) IL MIDI: MIDI/<record_time>.mid  (registrato direttamente dal pianoforte)
# ======================================================================
midi_path = ROOT / "MIDI" / (record_time + ".mid")
if midi_path.is_file():
    title("MIDI/%s  ->  %s" % (midi_path.name, human(midi_path.stat().st_size)))
    try:
        import mido

        mid = mido.MidiFile(str(midi_path))
        print("Formato SMF      : type %d, %d tracce, %d ticks per beat"
              % (mid.type, len(mid.tracks), mid.ticks_per_beat))
        print("Durata           : %.2f s" % mid.length)
        tempi = sorted({round(msg.tempo / 1e6, 3) for msg in mid if msg.type == "set_tempo"})
        print("Tempi            : %s %s" % (tempi[:5], "..." if len(tempi) > 5 else ""))
        programmi = sorted({msg.program for msg in mid if msg.type == "program_change"})
        print("Programmi MIDI   : %s" % programmi)
        note_on = [msg for msg in mid if msg.type == "note_on" and msg.velocity > 0]
        altezze = [msg.note for msg in note_on]
        print("Note suonate     : %d" % len(note_on))
        if altezze:
            print("Altezze          : %s - %s (MIDI %d - %d)"
                  % (pitch_name(min(altezze)), pitch_name(max(altezze)),
                     min(altezze), max(altezze)))
        print()
        print("Prime 5 note_on del file (tempo assoluto, altezza, velocity):")
        tempo_assoluto = 0.0
        trovate = 0
        for msg in mid:
            tempo_assoluto += msg.time
            if msg.type == "note_on" and msg.velocity > 0:
                print("  %8.4f s   %-4s (MIDI %3d)   velocity %3d"
                      % (tempo_assoluto, pitch_name(msg.note), msg.note, msg.velocity))
                trovate += 1
                if trovate == 5:
                    break
    except ImportError:
        print("Per i dettagli del MIDI installa mido:   %pip install -q mido")
else:
    print("MIDI/%s.mid non presente in %s" % (record_time, ROOT))

In [ ]:
# ======================================================================
# 6) SCHELETRO DELLE MANI: Handskeleton/<record_time>.json
# ======================================================================
if "ROOT" not in globals() or "record_time" not in globals():
    raise RuntimeError("Prima esegui la cella 0 (CONFIGURAZIONE) e la cella 2 (METADATA): "
                       "da li' arrivano ROOT e record_time. Poi riesegui questa cella.")
hs_path = file_modalita(ROOT, "Handskeleton/%s.json" % record_time, 120)
if hs_path is not None:
    title("Handskeleton/%s  ->  %s" % (hs_path.name, human(hs_path.stat().st_size)))
    if not LOAD_HAND_SKELETON:
        print("Caricamento saltato (LOAD_HAND_SKELETON = False): sono ~120 MB di JSON")
        print("per registrazione. Metti True in cima al notebook per analizzarlo.")
        print("Struttura: {'<indice_frame>': {'Left': [[x, y, z] x 21] oppure None,")
        print("                            'Right': [[x, y, z] x 21] oppure None}, ...}")
        print("21 punti per mano (formato MediaPipe Hands), coordinate normalizzate 0-1.")
    else:
        skeleton = json.loads(hs_path.read_text(encoding="utf-8"))
        frames = sorted(skeleton, key=int)
        print("Frame presenti   : %d (dal %s al %s)" % (len(frames), frames[0], frames[-1]))
        print("Video a 60 fps   : l'ultimo frame corrisponde a %.2f s"
              % (int(frames[-1]) / 60.0))
        primo = skeleton[frames[0]]
        print("Chiavi per frame :", list(primo))
        for mano in ["Left", "Right"]:
            punti = primo[mano]
            if punti is None:
                print("  %-5s: None (mano non rilevata in questo frame)" % mano)
            else:
                print("  %-5s: %d punti, polso [x, y, z] = [%.5f, %.5f, %.5f]"
                      % (mano, len(punti), punti[0][0], punti[0][1], punti[0][2]))
    print()
    print("Nota: i frame sono indicizzati sul video (60 fps), l'audio e' a 44.1 kHz.")

In [ ]:
# ======================================================================
# 7) L'AUDIO: Audio/<record_time>.wav  (WAV non compresso, 44.1 kHz)
# ======================================================================
if "ROOT" not in globals() or "record_time" not in globals():
    raise RuntimeError("Prima esegui la cella 0 (CONFIGURAZIONE) e la cella 2 (METADATA): "
                       "da li' arrivano ROOT e record_time. Poi riesegui questa cella.")
audio_path = file_modalita(ROOT, "Audio/%s.wav" % record_time, 65)
if audio_path is not None:
    title("Audio/%s  ->  %s" % (audio_path.name, human(audio_path.stat().st_size)))
    with wave.open(str(audio_path)) as wav:
        sample_rate = wav.getframerate()
        channels = wav.getnchannels()
        n_samples = wav.getnframes()
        sample_width = wav.getsampwidth()
    durata = n_samples / float(sample_rate)
    print("Sample rate     : %d Hz" % sample_rate)
    print("Canali          : %d %s" % (channels, "(mono)" if channels == 1 else "(stereo)"))
    print("Bit per campione: %d" % (8 * sample_width))
    print("Campioni        : %d" % n_samples)
    print("Durata          : %.2f s (%s)" % (durata, datetime.timedelta(seconds=round(durata))))
    print("Durata in metadata.json : %s" % record["duration"])
    if SHOW_AUDIO_EXCERPT:
        if "notes" in globals():
            start = max(0.0, float(notes["onset"].iloc[0]) - 0.5)
        else:
            start = 0.0
            print("(Parto da 0 s: se esegui prima la cella 3 (TSV) l'estratto comincia dalla")
            print(" prima nota, che di solito non e' a inizio registrazione.)")
        estratto = Path("/tmp") / (record_time + "_estratto.mp3")
        comando = ["ffmpeg", "-y", "-loglevel", "error", "-ss", "%.3f" % start,
                   "-t", str(EXCERPT_SECONDS), "-i", str(audio_path),
                   "-ac", "1", "-b:a", "128k", str(estratto)]
        try:
            from IPython.display import Audio, display

            subprocess.run(comando, check=True)
            print()
            print("Estratto di %d s da t = %.2f s:" % (EXCERPT_SECONDS, start))
            display(Audio(str(estratto)))
        except Exception as exc:
            print("Estratto non creato (%s)." % exc)
            print("Servono ffmpeg e IPython: in Colab ci sono entrambi.")
            print("File completo: %s" % audio_path)

In [ ]:
# ======================================================================
# 8) IL VIDEO: Video/<record_time>.mp4  (1920x1080, 60 fps, H.264 + AAC)
# ======================================================================
if "ROOT" not in globals() or "record_time" not in globals():
    raise RuntimeError("Prima esegui la cella 0 (CONFIGURAZIONE) e la cella 2 (METADATA): "
                       "da li' arrivano ROOT e record_time. Poi riesegui questa cella.")
video_path = file_modalita(ROOT, "Video/%s.mp4" % record_time, 324)
if video_path is not None:
    title("Video/%s  ->  %s" % (video_path.name, human(video_path.stat().st_size)))
    try:
        comando = ["ffprobe", "-v", "error", "-print_format", "json",
                   "-show_streams", "-show_format", str(video_path)]
        info = json.loads(subprocess.run(comando, check=True, capture_output=True,
                                         text=True).stdout)
        for stream in info["streams"]:
            if stream["codec_type"] == "video":
                num, den = stream["avg_frame_rate"].split("/")
                print("Video : %s %sx%s, %.2f fps, %s frame"
                      % (stream["codec_name"], stream["width"], stream["height"],
                         float(num) / float(den), stream.get("nb_frames", "?")))
            else:
                print("Audio : %s, %s Hz, %s canali"
                      % (stream["codec_name"], stream.get("sample_rate"),
                         stream.get("channels")))
        print("Durata: %.2f s" % float(info["format"]["duration"]))
    except Exception as exc:
        print("ffprobe non disponibile (%s)" % exc)
    record_corrente = globals().get("record") or {}
    nomi = [str(record_corrente.get("P1_name")), str(record_corrente.get("P2_name"))]
    if any("jiwoo" in nome.lower() for nome in nomi):
        print("NB: in questa registrazione il busto della pianista e' sfocato per privacy;")
        print("    tastiera e mani sono visibili.")
    if SHOW_VIDEO:
        # Il file intero pesa centinaia di MB: metterlo nell'output del notebook lo farebbe
        # esplodere. Mostriamo un estratto dei primi VIDEO_EXCERPT_SECONDS secondi, a 640 px.
        estratto = Path(tempfile.gettempdir()) / (record_time + "_video_estratto.mp4")
        comando = ["ffmpeg", "-y", "-loglevel", "error", "-ss", "0", "-t",
                   str(VIDEO_EXCERPT_SECONDS), "-i", str(video_path),
                   "-vf", "scale=640:-2", "-c:v", "libx264", "-preset", "veryfast",
                   "-crf", "30", "-c:a", "aac", "-b:a", "64k", str(estratto)]
        try:
            from IPython.display import Video, display

            subprocess.run(comando, check=True)
            print()
            print("Estratto dei primi %d s a 640 px (%s):"
                  % (VIDEO_EXCERPT_SECONDS, human(estratto.stat().st_size)))
            display(Video(str(estratto), embed=True, width=640))
            print("File completo: %s" % video_path)
        except Exception as exc:
            print("Anteprima non creata (%s)." % exc)
            print("Servono ffmpeg e IPython: in Colab ci sono entrambi.")
            print("Puoi comunque aprire il file completo a mano: %s" % video_path)
    else:
        print("(Anteprima video non mostrata: SHOW_VIDEO = False.)")

In [ ]:
# ======================================================================
# 9) RIEPILOGO: tutti i file di questa riga del dataset
# ======================================================================
elenco = [("Audio", "wav"), ("Video", "mp4"), ("Handskeleton", "json"),
          ("MIDI", "mid"), ("TSV", "tsv"), ("Fingering", "tsv"), ("Fingering_GT", "tsv")]
righe = []
for cartella, estensione in elenco:
    percorso = ROOT / cartella / (record_time + "." + estensione)
    righe.append({"modality": cartella,
                  "file": cartella + "/" + percorso.name,
                  "presente": "si" if percorso.is_file() else "no",
                  "dimensione": human(percorso.stat().st_size) if percorso.is_file() else "-"})
print(pd.DataFrame(righe).to_string(index=False))
mancanti = [r["file"] for r in righe if r["presente"] == "no"]
if mancanti:
    print()
    print("Non ancora sul disco:", ", ".join(mancanti))
    print("Le celle di quelle modalita' se li scaricano da sole quando le esegui")
    print("(AUTO_DOWNLOAD_MEDIA = True). Per averli tutti subito: DOWNLOAD_MEDIA = True e")
    print("DOWNLOAD_SKELETON = True nella cella 0, poi riesegui la cella DATASET.")

title("UN ESEMPIO DI RIGA DEL DATASET: %s" % record_time)
print("  split            : %s" % record["split"])
print("  brano            : %s - %s" % (record["composer"], record["piece"]))
print("  esecuzione       : %s / %s" % (record["performance_method"],
                                         record["performance_type"]))
print("  pianista         : %s (%s, %s, %s, music major: %s)"
      % (record["P1_name"], record["P1_gender"], record["P1_age"], record["P1_skill"],
         record["P1_musicmajor"]))
print("  durata           : %s" % record["duration"])
print("  note nel TSV     : %d" % len(notes))
if fing_path.is_file():
    print("  note diteggiate  : %d/%d (%.1f%%)"
          % (len(labeled), len(fingering), 100.0 * len(labeled) / len(fingering)))
print("  prima nota       : t = %.4f s, %s (MIDI %d), velocity %d"
      % (notes["onset"].iloc[0], pitch_name(notes["note"].iloc[0]),
         notes["note"].iloc[0], notes["velocity"].iloc[0]))

## Note utili

**Come sono fatti i file**

| Cartella | Cosa contiene | Formato |
|---|---|---|
| `metadata.json` | le 107 registrazioni con split, brano, compositore, pianista, durata | JSON, dict `{"0": {...}, "1": {...}}` |
| `TSV/` | una riga per nota: `onset`, `key_offset`, `frame_offset`, `note`, `velocity` | TSV (header commentato con `#`) |
| `Fingering/` | le stesse righe del `TSV/` piu' `hand` (L/R/Noinfo) e `finger` (1-5/Noinfo) | TSV |
| `Fingering_GT/` | diteggiatura annotata a mano, solo 11 registrazioni (1.800 note) | TSV |
| `MIDI/` | performance ground truth registrata dal pianoforte digitale | Standard MIDI File |
| `Handskeleton/` | 21 landmark per mano, frame per frame | JSON (~120 MB per registrazione) |
| `Audio/` | audio grezzo mono, 44.1 kHz | WAV (~65 MB per registrazione) |
| `Video/` | tastiera e mani, 1920x1080 a 60 fps | MP4 (~340 MB per registrazione) |

In tutte le cartelle il nome del file e' lo stesso: il `<record_time>` di `metadata.json`
(es. `2024-02-14_19-10-09`). Le `Fingering/` hanno le righe allineate 1:1 con le `TSV/`.

**Attenzione allo snippet del dataset card.** Il README del dataset mostra
`load_dataset("PianoVAM/PianoVAM_v1")` con colonne `audio_path`, `video_path`, `midi_path`:
quello era il builder della v1.0. Nell'ultima revisione del repo la config predefinita si
chiama `midi_notes` ed e' solo l'unione dei `TSV/` (5 colonne, 527.302 righe), senza colonne
di percorso. Per l'esempio multimodale completo servono `metadata.json` + le cartelle, cioe'
esattamente quello che fa questo notebook.

**Diteggiatura.** I file `Fingering/` sono etichette automatiche: circa il 20% delle note
e' `Noinfo` (la percentuale varia dal 2,7% all'81,4% per registrazione, mediana 15,5%).
Sulle note etichettate l'accuratezza misurata contro `Fingering_GT/` e' del 95,7% per
mano + dito e del 99,2% per la sola mano.

**Licenza e citazione.** Dataset CC BY-NC-SA 4.0 (solo uso non commerciale):

```bibtex
@inproceedings{kim2025pianovam,
  title={PianoVAM: A Multimodal Piano Performance Dataset},
  author={Kim, Yonghyun and Park, Junhyung and Bae, Joonhyung and Kim, Kirak and Kwon, Taegyun and Lerch, Alexander and Nam, Juhan},
  booktitle={Proceedings of the 26th International Society for Music Information Retrieval Conference (ISMIR)},
  year={2025}
}
```